# Probability and Statistics

## Learning Objectives
1. Derive MLE and MAP parameter estimates for a Gaussian distribution using NumPy.
2. Apply t-test, chi-square test, and bootstrap confidence intervals with scipy.
3. Run an A/B test significance analysis on simulated conversion data.
4. Demonstrate Bayesian updating and calibration of probabilistic classifiers.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy import stats
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: MLE and MAP for Gaussian Parameters (NumPy)


In [ ]:
def mle_gaussian(data: np.ndarray):
    """Maximum Likelihood Estimates for mu and sigma^2."""
    n = len(data)
    mu_mle = data.mean()
    sigma2_mle = ((data - mu_mle) ** 2).mean()
    return mu_mle, sigma2_mle


def map_gaussian_mu(data: np.ndarray, mu0: float, sigma0: float,
                    sigma_known: float):
    """MAP estimate of mu given Gaussian prior N(mu0, sigma0^2) and known sigma."""
    n = len(data)
    sigma2 = sigma_known ** 2
    sigma0_2 = sigma0 ** 2
    sigma_post2 = 1.0 / (n / sigma2 + 1.0 / sigma0_2)
    mu_map = sigma_post2 * (data.sum() / sigma2 + mu0 / sigma0_2)
    return mu_map, np.sqrt(sigma_post2)


true_mu, true_sigma = 3.5, 1.0
data = np.random.normal(true_mu, true_sigma, size=20)

mu_mle, sigma2_mle = mle_gaussian(data)
mu_map, sigma_post = map_gaussian_mu(data, mu0=0.0, sigma0=2.0, sigma_known=true_sigma)

print(f"True mu               : {true_mu:.4f}")
print(f"MLE mu                : {mu_mle:.4f}  (sigma^2={sigma2_mle:.4f})")
print(f"MAP mu                : {mu_map:.4f}  (posterior sigma={sigma_post:.4f})")

print("\nConvergence: MAP approaches MLE as n increases")
print(f"{'n':>6} | {'MLE mu':>10} | {'MAP mu':>10} | {'Delta':>8}")
print("-" * 38)
for n_demo in [5, 10, 20, 50, 200, 1000]:
    d = np.random.normal(true_mu, true_sigma, n_demo)
    m_mle, _ = mle_gaussian(d)
    m_map, _ = map_gaussian_mu(d, 0.0, 2.0, true_sigma)
    print(f"{n_demo:>6} | {m_mle:>10.4f} | {m_map:>10.4f} | {abs(m_mle-m_map):>8.4f}")


## Level 2: t-test / chi-square / bootstrap (scipy)


In [ ]:
# --- 1. Two-sample t-test ---
group_a = np.random.normal(5.0, 1.5, 60)
group_b = np.random.normal(5.6, 1.5, 60)
t_stat, p_value = stats.ttest_ind(group_a, group_b)
print(f"Two-sample t-test: t={t_stat:.3f}, p={p_value:.4f}")
print(f"  Reject H0 (alpha=0.05): {p_value < 0.05}")

# --- 2. Chi-square test for independence ---
observed = np.array([[40, 20], [25, 35]])
chi2, p_chi, dof, expected = stats.chi2_contingency(observed)
print(f"\nChi-square test: chi2={chi2:.3f}, p={p_chi:.4f}, dof={dof}")
print(f"  Reject H0 (independence) at alpha=0.05: {p_chi < 0.05}")

# --- 3. Bootstrap confidence interval ---
def bootstrap_ci(data: np.ndarray, stat_fn=np.mean,
                  n_boot: int = 2000, alpha: float = 0.05) -> tuple:
    """Percentile bootstrap CI for a statistic."""
    boot_stats = np.array([stat_fn(np.random.choice(data, len(data)))
                            for _ in range(n_boot)])
    lo = np.percentile(boot_stats, 100 * alpha / 2)
    hi = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    return lo, hi


sample_data = np.random.exponential(scale=2.0, size=100)
true_mean = 2.0
ci_lo, ci_hi = bootstrap_ci(sample_data)
print(f"\nBootstrap 95% CI for mean: [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"True mean {true_mean} inside CI: {ci_lo <= true_mean <= ci_hi}")
print(f"Sample mean: {sample_data.mean():.4f}")
# Note: for GPU computations, wrap in try/except RuntimeError to catch
# out of memory errors: reduce n_boot or sample size if OOM occurs.


## Real-World Example 1: A/B Test Significance Analysis


In [ ]:
from scipy.stats import norm

np.random.seed(7)
n_a, n_b = 1000, 1000
conv_rate_a = 0.08
conv_rate_b = 0.10

conversions_a = np.random.binomial(n_a, conv_rate_a)
conversions_b = np.random.binomial(n_b, conv_rate_b)

print(f"Group A: {conversions_a}/{n_a} ({100*conversions_a/n_a:.2f}%)")
print(f"Group B: {conversions_b}/{n_b} ({100*conversions_b/n_b:.2f}%)")

p_a_hat = conversions_a / n_a
p_b_hat = conversions_b / n_b
p_pooled = (conversions_a + conversions_b) / (n_a + n_b)
se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_a + 1/n_b))
z_stat = (p_b_hat - p_a_hat) / se
p_value_ab = 2 * (1 - norm.cdf(abs(z_stat)))

se_diff = np.sqrt(p_a_hat*(1-p_a_hat)/n_a + p_b_hat*(1-p_b_hat)/n_b)
ci_lo_ab = (p_b_hat - p_a_hat) - 1.96 * se_diff
ci_hi_ab = (p_b_hat - p_a_hat) + 1.96 * se_diff

print(f"\nz-statistic : {z_stat:.4f}")
print(f"p-value     : {p_value_ab:.4f} (two-tailed)")
print(f"95% CI for (p_B - p_A): [{ci_lo_ab:.4f}, {ci_hi_ab:.4f}]")
print(f"Significant at 0.05: {p_value_ab < 0.05}")


def compute_power(n, p_a, p_b, alpha=0.05):
    """Estimate power of two-proportion z-test."""
    p_pool = (p_a + p_b) / 2
    se_h0 = np.sqrt(2 * p_pool * (1 - p_pool) / n)
    se_h1 = np.sqrt(p_a * (1-p_a)/n + p_b * (1-p_b)/n)
    z_crit = norm.ppf(1 - alpha / 2)
    power = 1 - norm.cdf(z_crit - (p_b - p_a) / se_h1)
    return power


sample_sizes_ab = [100, 200, 500, 1000, 2000, 5000]
print("\nSample size vs power (alpha=0.05):")
for n in sample_sizes_ab:
    pwr = compute_power(n, conv_rate_a, conv_rate_b)
    print(f"  n={n:5d}: power={pwr:.3f}")


## Real-World Example 2: Bayesian Updating


In [ ]:
theta_grid = np.linspace(0, 1, 300)
true_theta = 0.65

observations = np.random.binomial(1, true_theta, size=100)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.ravel()
n_after = [0, 5, 10, 20, 50, 100]

for ax, n in zip(axes, n_after):
    heads = observations[:n].sum()
    tails = n - heads
    alpha_post = 1.0 + heads
    beta_post = 1.0 + tails
    posterior_pdf = stats.beta.pdf(theta_grid, alpha_post, beta_post)
    posterior_mean = alpha_post / (alpha_post + beta_post)

    ax.plot(theta_grid, posterior_pdf, color='steelblue', linewidth=2)
    ax.axvline(true_theta, color='red', linestyle='--',
               label=f'True theta={true_theta}')
    ax.axvline(posterior_mean, color='navy', linestyle=':',
               label=f'Post. mean={posterior_mean:.3f}')
    ax.fill_between(theta_grid, posterior_pdf, alpha=0.2, color='steelblue')
    ax.set_title(f'n={n} (H={heads}, T={tails})')
    ax.set_xlabel('theta'); ax.set_ylabel('Density')
    ax.legend(fontsize=7); ax.set_xlim(0, 1)

plt.suptitle('Bayesian Updating: Beta-Binomial Coin Bias Inference', fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/bayesian_updating.png', dpi=80)
plt.close()
print('Bayesian updating plot saved to /tmp/bayesian_updating.png')
print(f'After 100 flips: posterior mean = {observations.sum()/100:.3f} (true: {true_theta})')


## Real-World Example 3: Calibration Plot for Probabilistic Classifiers


In [ ]:
X_cal, y_cal = make_classification(n_samples=3000, n_features=20,
                                    n_informative=10, random_state=42)
X_tr_cal, X_te_cal, y_tr_cal, y_te_cal = train_test_split(
    X_cal, y_cal, test_size=0.4, random_state=42
)

lr_model = LogisticRegression(max_iter=300).fit(X_tr_cal, y_tr_cal)
prob_lr = lr_model.predict_proba(X_te_cal)[:, 1]

svc_cal = CalibratedClassifierCV(LinearSVC(max_iter=1000), cv=3, method='sigmoid')
try:
    svc_cal.fit(X_tr_cal, y_tr_cal)
    prob_svc_cal = svc_cal.predict_proba(X_te_cal)[:, 1]
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        print("OOM: using LR probabilities as fallback")
        prob_svc_cal = prob_lr
    else:
        raise

n_bins = 10
frac_pos_lr,  mean_pred_lr  = calibration_curve(y_te_cal, prob_lr,
                                                  n_bins=n_bins)
frac_pos_svc, mean_pred_svc = calibration_curve(y_te_cal, prob_svc_cal,
                                                  n_bins=n_bins)

brier_lr  = np.mean((prob_lr - y_te_cal) ** 2)
brier_svc = np.mean((prob_svc_cal - y_te_cal) ** 2)
print(f"Logistic Regression Brier score  : {brier_lr:.4f}")
print(f"Calibrated SVC Brier score       : {brier_svc:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.plot(mean_pred_lr,  frac_pos_lr,  marker='o', color='steelblue',
        linewidth=2, label=f'Logistic Reg (Brier={brier_lr:.3f})')
ax.plot(mean_pred_svc, frac_pos_svc, marker='s', color='coral',
        linewidth=2, label=f'Calibrated SVC (Brier={brier_svc:.3f})')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curve (Reliability Diagram)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/calibration_plot.png', dpi=80)
plt.close()
print('Calibration plot saved to /tmp/calibration_plot.png')
